# Name That LLM — Qwen3-8B-Base QLoRA sequence classifier (training)

Fine-tunes Qwen3-8B-Base (4-bit NF4) with LoRA for 18-way authorship attribution.
Validation holds out whole `prompt_id` groups. Outputs (in `/kaggle/working`):
`adapter/` (LoRA adapter + `score` head weights, plus `label_map.json`) and `submission.parquet`.

In [ ]:
%pip install -q bitsandbytes

In [ ]:
import os

# Pin this process to the first T4 so the Trainer sees exactly one GPU
# (with two visible GPUs it doubles the per-step batch and rescales the loss
# even though a 4-bit model is never wrapped in DataParallel). GPU 1 stays free.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time

RUN_START = time.time()

SMOKE_TEST = False  # True: tiny subset, 1 epoch, for pipeline debugging only

SEED = 42
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 2  # smoke run on the 64 longest texts peaked at 7.5 GB reserved on a 15 GB T4
GRAD_ACCUM_STEPS = 8  # effective batch = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS = 16
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
VAL_SIZE = 0.15

DATA_DIR = "/kaggle/input/competitions/name-that-llm"
MODEL_PATH = (
    "/kaggle/input/models/qwen-lm/qwen-3/"
    "transformers/8b-base/1"
)
OUTPUT_DIR = "/kaggle/working"
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
CHECKPOINT_DIR = "/tmp/checkpoints"  # outside /kaggle/working; deleted after training

In [ ]:
import subprocess
import torch

print("=== GPU ===")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used",
     "--format=csv"],
    capture_output=True, text=True,
).stdout)
print("CUDA available:", torch.cuda.is_available())
print("Visible to this process:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print("Memory (GB):", round(
            torch.cuda.get_device_properties(i).total_memory / 1e9, 2
        ))

print("\n=== INPUT FILES ===")
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for filename in files[:5]:
        print("  ", filename)

In [ ]:
import pandas as pd

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_parquet(
    f"{DATA_DIR}/sample_submission.parquet"
)

print("=== DATASET SHAPES ===")
print("Train:", train.shape)
print("Test:", test.shape)

print("\n=== TRAINING COLUMNS ===")
print(train.columns.tolist())

print("\n=== FIRST TRAINING SAMPLE ===")
print(train.iloc[0].to_dict())

print("\n=== MODEL DISTRIBUTION ===")
print(train["model"].value_counts())

print("\n=== SUBMISSION FORMAT ===")
print(sample.head())
print(sample.dtypes)

In [ ]:
import importlib.metadata as metadata

print("=== LIBRARY VERSIONS ===")

for package in [
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "scikit-learn",
]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

print("\n=== GPU CAPABILITIES ===")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}")
    print("Compute capability:", props.major, props.minor)

print("\n=== QWEN TOKENIZER ===")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

sample_text = train.iloc[0]["text"]
tokens = tokenizer(sample_text)

print("Tokenizer loaded successfully!")
print("Number of tokens:", len(tokens["input_ids"]))
print("Vocabulary size:", len(tokenizer))
print("Padding token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

In [ ]:
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print("bitsandbytes version:", bnb.__version__)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit configuration created successfully!")
print(quantization_config)

In [ ]:
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(SEED)

# Consistent mapping between model names and class indices
model_names = sorted(train["model"].unique())

label2id = {
    name: i for i, name in enumerate(model_names)
}

id2label = {
    i: name for name, i in label2id.items()
}

print("Number of classes:", len(model_names))

# Load the pretrained model in 4-bit precision, pinned to a single GPU
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=18,
    id2label=id2label,
    label2id=label2id,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    local_files_only=True,
)

model.config.pad_token_id = tokenizer.pad_token_id

print("\nModel loaded successfully!")
print("Model type:", type(model).__name__)
print("Parameter devices:", {str(p.device) for p in model.parameters()})

# Check GPU memory use
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9

    print(
        f"GPU {i}: {allocated:.2f} GB allocated, "
        f"{reserved:.2f} GB reserved"
    )

In [ ]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Prepare the quantized Qwen model for training
model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# Define our first experiment's LoRA parameters
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    modules_to_save=["score"],
    bias="none"
)

# Attach trainable adapters
model = get_peft_model(model, lora_config)

print("LoRA adapters attached!\n")

model.print_trainable_parameters()

print("\n=== CLASSIFICATION HEAD ===")
for name, param in model.named_parameters():
    if "score" in name:
        print(name, "Trainable:", param.requires_grad)

print("\n=== GPU MEMORY ===")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated"
    )

## Dataset, prompt-grouped split, and token lengths

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit

df = train[["id", "prompt_id", "text", "model"]].copy()
df["labels"] = df["model"].map(label2id).astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, groups=df["prompt_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

assert not set(train_df["prompt_id"]) & set(val_df["prompt_id"])
print(f"Train: {len(train_df)} rows / {train_df['prompt_id'].nunique()} prompts")
print(f"Val:   {len(val_df)} rows / {val_df['prompt_id'].nunique()} prompts")
print("Val prompt_ids:", sorted(val_df["prompt_id"].unique()))
print("\nVal class counts:")
print(val_df["model"].value_counts().sort_index().to_string())

if SMOKE_TEST:
    # Longest texts first so the smoke run also probes peak memory.
    train_df = (train_df.assign(n=train_df["text"].str.len())
                .sort_values("n", ascending=False).head(64).drop(columns="n")
                .reset_index(drop=True))
    val_df = val_df.groupby("labels").head(2).reset_index(drop=True)
    NUM_EPOCHS = 1
    print(f"\nSMOKE_TEST: train={len(train_df)}, val={len(val_df)}, epochs={NUM_EPOCHS}")

train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
print(train_ds)
print(val_ds)

In [ ]:
# Token-length distribution over all training texts (untruncated) to justify MAX_LENGTH
lengths = np.array([len(ids) for ids in tokenizer(train["text"].tolist())["input_ids"]])

print("=== TOKEN LENGTHS (train.csv, untruncated) ===")
print(f"min={lengths.min()}  mean={lengths.mean():.1f}  max={lengths.max()}")
for p in (50, 90, 95, 99, 100):
    print(f"p{p}: {np.percentile(lengths, p):.0f}")

bins = list(range(0, max(MAX_LENGTH, int(lengths.max())) + 64, 64))
counts, edges = np.histogram(lengths, bins=bins)
scale = 60 / counts.max()
print("\nHistogram (64-token bins):")
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{int(lo):4d}-{int(hi) - 1:4d} | {'#' * int(round(c * scale)):<60} {c}")

n_trunc = int((lengths > MAX_LENGTH).sum())
print(f"\nMAX_LENGTH={MAX_LENGTH}: {n_trunc} / {len(lengths)} texts "
      f"({100 * n_trunc / len(lengths):.2f}%) would be truncated")

## Tokenize and train

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import balanced_accuracy_score


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {"balanced_accuracy": balanced_accuracy_score(labels, preds)}


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",
    greater_is_better=True,
    logging_steps=10,
    label_names=["labels"],
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer n_gpu:", training_args.n_gpu)
print("Effective batch size:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS * max(1, training_args.n_gpu))

In [ ]:
torch.cuda.reset_peak_memory_stats()
train_start = time.time()

train_result = trainer.train()

train_minutes = (time.time() - train_start) / 60
print(train_result)
print(f"Training wall time: {train_minutes:.1f} min")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.max_memory_reserved() / 1e9:.2f} GB reserved")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best balanced_accuracy:", trainer.state.best_metric)

print("\n=== EPOCH EVAL HISTORY ===")
for entry in trainer.state.log_history:
    if "eval_balanced_accuracy" in entry:
        print(f"epoch {entry['epoch']:.2f}: eval_loss={entry['eval_loss']:.4f} "
              f"balanced_accuracy={entry['eval_balanced_accuracy']:.4f}")

## Held-out prompt groups: balanced accuracy and per-class recall (best checkpoint)

In [ ]:
from sklearn.metrics import recall_score

val_out = trainer.predict(val_tok)
val_logits = val_out.predictions[0] if isinstance(val_out.predictions, tuple) else val_out.predictions
val_preds = np.argmax(val_logits, axis=-1)
val_labels = val_out.label_ids

val_bal_acc = balanced_accuracy_score(val_labels, val_preds)
recalls = recall_score(val_labels, val_preds, labels=list(range(len(model_names))),
                       average=None, zero_division=0)
support = np.bincount(val_labels, minlength=len(model_names))

print(f"Validation balanced accuracy (held-out prompts): {val_bal_acc:.4f}")
print(f"Validation accuracy: {(val_preds == val_labels).mean():.4f}")
print("\nPer-class recall:")
print(f"{'model':<26}{'recall':>8}{'support':>9}")
for i, name in enumerate(model_names):
    print(f"{name:<26}{recalls[i]:>8.3f}{support[i]:>9d}")

## Save LoRA adapter + classification head only

In [ ]:
import json
import shutil

model.save_pretrained(ADAPTER_DIR)
with open(f"{ADAPTER_DIR}/label_map.json", "w") as f:
    json.dump({"id2label": {str(i): n for i, n in id2label.items()}}, f, indent=2)

shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

from safetensors import safe_open

with safe_open(f"{ADAPTER_DIR}/adapter_model.safetensors", "pt") as f:
    keys = list(f.keys())
print(f"{len(keys)} tensors saved; score head tensors:",
      [k for k in keys if "score" in k])

for root, _, files in os.walk(ADAPTER_DIR):
    for fn in files:
        p = os.path.join(root, fn)
        print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p}")

## Predict test.csv and write submission.parquet

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq


@torch.no_grad()
def predict_labels(texts, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    order = np.argsort([len(t) for t in texts])[::-1]
    pred_ids = np.empty(len(texts), dtype=int)
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        enc = tokenizer([texts[i] for i in idx], truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors="pt").to(model.device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        pred_ids[idx] = logits.argmax(-1).cpu().numpy()
    return [id2label[int(i)] for i in pred_ids]


test_preds = predict_labels(test["text"].tolist())
submission = pd.DataFrame({"id": test["id"], "model": test_preds})

sample_schema = pq.read_schema(f"{DATA_DIR}/sample_submission.parquet")
table = pa.Table.from_pandas(submission, preserve_index=False).cast(sample_schema)
pq.write_table(table, f"{OUTPUT_DIR}/submission.parquet")

written = pq.read_table(f"{OUTPUT_DIR}/submission.parquet")
assert written.schema.equals(sample_schema, check_metadata=False), (written.schema, sample_schema)
assert written.num_rows == len(test)
assert set(written.column("id").to_pylist()) == set(test["id"])
assert set(written.column("model").to_pylist()) <= set(model_names)
print(written.schema)
print(written.to_pandas().head(10))
print("Prediction counts:", pd.Series(test_preds).value_counts().to_dict())

In [ ]:
print(f"Total notebook runtime: {(time.time() - RUN_START) / 60:.1f} min")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used",
     "--format=csv"],
    capture_output=True, text=True,
).stdout)
print("Output dir:", sorted(os.listdir(OUTPUT_DIR)))